<a href="https://colab.research.google.com/github/yuchanleee/iot-ids-ml/blob/main/Ids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. 데이터 전처리**

1-1) 데이터 다운로드 및 특징 확인

In [ ]:
# 라이브러리 import

import os
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style='darkgrid')

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve, auc, accuracy_score
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, learning_curve
from scipy.stats import randint,uniform

In [ ]:
# 데이터 다운로드 및 경로 확인
import kagglehub

path = kagglehub.dataset_download("amineipad/ids-iot-dl-dataset")
print("Path to dataset files:", path)

100%|██████████| 297M/297M [00:02<00:00, 131MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/amineipad/ids-iot-dl-dataset/versions/1


In [ ]:
# df라는 변수에 csv파일 불러와서 저장
files = os.listdir(path)
for f in files:
  print(f)

file_path = os.path.join(path, "DNN-EdgeIIoT-dataset.csv")
df = pd.read_csv(file_path)
df.info()

DNN-EdgeIIoT-dataset.csv


/tmp/ipykernel_5917/3551689585.py:7: DtypeWarning: Columns (2,3,6,11,13,14,15,16,17,31,32,34,39,45,51,54,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2219201 entries, 0 to 2219200
Data columns (total 63 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   frame.time                 object 
 1   ip.src_host                object 
 2   ip.dst_host                object 
 3   arp.dst.proto_ipv4         object 
 4   arp.opcode                 float64
 5   arp.hw.size                float64
 6   arp.src.proto_ipv4         object 
 7   icmp.checksum              float64
 8   icmp.seq_le                float64
 9   icmp.transmit_timestamp    float64
 10  icmp.unused                float64
 11  http.file_data             object 
 12  http.content_length        float64
 13  http.request.uri.query     object 
 14  http.request.method        object 
 15  http.referer               object 
 16  http.request.full_uri      object 
 17  http.request.version       object 
 18  http.response              float64
 19  http.tls_port              float64
 20  tc

In [ ]:
# @title
기본정보
1. frame.time - 패킷 캡쳐된 절대시각
2. ip.src_host - 출발지 ip 주소
3. ip.dst_host - 목적지 ip 주소

ARP - ip주소를 mac 주소로 바꿀 때 사용하는 프로토콜 (공유기 및 내부 네트워크에서만 사용)

4. arp.dst.proto_ipv4 - 목적지 ipv4 주소
5. arp.opcode - arp 메세지 유형 ( 1- arp request, 브로드캐스트 2 - arp reply, 유니캐스트)
6. arp.hw.size - mac 주소 길이. 이더넷(유선통신), wifi와bluetooth 등 대부분의 L2 프로토콜에서 6바이트 통일

7. arp.src.proto_ipv4 - 출발지 ipv4 주소

ICMP - 인터넷 제어 메세지 프로토콜 (오류 보고 및 네트워크 상태 확인, L3 계층)

8. icmp.checksum - 데이터 무결성을 검사하는 체크섬 값 (type + Code + Rest of header + Data 16진수 계산 -> 오버플로우 처리 -> 비트 반전)
9. icmp.seq_le - 핑 번호, 리틀엔디안 순서
10. icmp.transmit_timestamp - 핑 보낸 시각
11. icmp.unused - 안쓰는 필드 0으로 채움.

http

12. http.file_data - http 메세지 Body 즉 payload. Get 요청일때는 0이고, Post 또는 html 및 이미지 응답일때는 안에 내용 들어감.
13. http.content_length - file_data 크기
14. http.request.uri.query - url 뒤에 붙는 쿼리 부분.
15. http.request.method - http 요청 방식 ex) post get put delete 등..
16. http.referer - 이 페이지로 오기 전에 있었던 페이지 url
17. http.request.full_uri - 전체 uri 즉 프로토콜+도메인+경로+쿼리 url
18. http.request.version - http 버전
19. http.response - 서버가 보낸 응답 ex) 200(정상) 301(리다이렉트) 등..
20. http.tls_port - tls용 포트번호

tcp/udp는 기본적으로 L3 ip헤더 프로토콜 필드에서 결정

tcp - L4계층 통신규약 프로토콜

21. tcp.ack - 32비트 시퀸스 번호, 보기쉽게 raw값 시작점 1로 변경. 받아야될 다음 번호 의미.
22. tcp.ack_raw - tcp.ack 번호 raw값. 시작점은 무작위 isn(initial sequence number) 골라 시작 (해킹방지)
23. tcp.checksum - tcp 단계 checksum연산 (icmp 참고)
24. tcp.connection.fin  - 연결 강제 종료 여부
25. tcp.connection.rst - 연결 정상 종료 여부
26. tcp.connection.syn - 클라이언트가 서버와 동기화 요청 여부 (isn 전달)
27. tcp.connection.synack - 서버가 syn받고 ack 전달 여부
28. tcp.dstport- 목적지 포트번호
29. tcp.flags - syn, rst, fin, ack등 한번에 묶어서 보여줌
30. tcp.flags.ack - ack 연결중인지 여부
31. tcp.len - tcp 상위계층 데이터 길이 (L4 상위 즉 http등..)
32. tcp.options - tcp헤더 뒤 추가 옵션 헤더. 가변 길이이며 크게 1. MSS (한번 전송 최대 데이터 크기) 2. Window Scale (tcp헤더의 크기는 원래 16비트. 하지만 네트워크가 발전하며 한번에 많이 데이터 보내기 위해 2^window scale 곱해서 공간 할당 가능하게 함) 3. Sack (빠진 패킷 번호만 재전송) 4. Timestamps (패킷 보낸 시간 정보) 존재
33. tcp.payload - tcp가 담고있는 내용들 (L4 상위계층 내용)
34. tcp.seq - 보내는 패킷이 시작하는 sequence 번호
35. tcp.srcport - 출발지 포트 번호

udp - L4계층 통신규약 프로토콜

36. udp.port - udp 통신시 사용 포트번호
37. udp.stream - 같은 ip와 포트 쌍마다 묶어서 임의로 부여한 스트림 번호
38. udp.time_delta - 같은 stream에서 이전 패킷과 현재 패킷 사이의 시간 지연

dns

39. dns.qry.name - 물어본 도메인 주소 이름
40. dns.qry.name.len - dns.qry.name 글자수
41. dns.qry.qu - mdns에서 0이면 멀티캐스트, 1이면 유니캐스트 응답 요청
42. dns.qry.type - 필요한 정보 타입 명시. 1이면 ipv4 28이면 ipv6 등..
43. dns.retransmission - 재전송된 패킷 판단 (wireshark가 임의로 붙인 것)
44. dns.retransmit_request - retransmission 중 요청 패킷 재전송 판단.
45. dns.retransmit_request_in - 응답 패킷 중 몇 번째 재전송 요청에 대한 응답인지 의미

mqtt - iot 프로토콜. OSI 7계층이며 http와 달리 중간에 브로커가 패킷 전달.

46. mqtt.conack.flags - 브로커가 클라이언트에게 알려주는 이전에 클라이언트 통신 정보 재사용 여부 (0:처음부터 1:재사용)
47. mqtt.conflag.cleansess - 클라이언트가 브로커에게 요청하는 이전 클라이언트 통신 재사용 여부 (0: 재사용 1:처음부터)
48. mqtt.conflags - connect 패킷 내부 초기 설정. 8비트로 구성되며 cleansession 1비트 wiiflag(연결끊길 시 브로커가 메세지 전달) 2비트 willqdos&retain 3-5비트 패스워드/아이디 스위치 6-7비트
49. mqtt.hdrflags - 모든 mqtt패킷 앞에 붙는 패킷 종류 및 설정. 앞 4비트는 패킷 종류, 뒤 4비트는 패킷 설정
50. mqtt.len - 헤더 제외 내용 길이
51. mqtt.msg_decoded_as - msg 내용 디코딩 방식
52. mqtt.msg - 실제 페이로드 데이터
53. mqtt.msgtype - mqtt 패킷 종류 (hdfflag 앞 4비트)
54. mqtt.proto_len - protoname 문자열 글자 수
55. mqtt.protoname - 통신에 사용할 프로토콜 이름 (여기선 mqtt)
56. mqtt.topic - 브로커가 메세지 분류시 사용하는 문자열 주소
57. mqtt.topic_len - topic 문자열 글자 수
58. mqtt.ver - mqtt 프로토콜 버전

mdtcp - modebus tcp의 줄임말이며, 산업용 기계에 많이 쓰임. 브로커 사용X 1:1 통신방식이며 연결안전성 중시.

59. mbtcp.len - 헤더 제외 내용 길이
60. mbtcp.trans_id - 장치 식별 번호
61. mbtcp.unit_id - 데이터 통신 일련번호

Attack - 공격분류

62. Attack_label - 공격여부
63. Attack_type - 공격종류

In [ ]:
#데이터 일부 확인
df.head()

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 11:44:10.081753000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 11:44:10.162218000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal
2,2021 11:44:10.162271000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
3,2021 11:44:10.162641000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 11:44:10.166132000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,Temperature_and_Humidity,24.0,0.0,0.0,0.0,0.0,0,Normal


In [ ]:
# null값 있는 칼럼 검사 위한 함수 생성 & 검사
def display_missing(df):
  for col in df.columns.tolist():
    print(f'{col} missing: {df[col].isnull().sum()}')

display_missing(df)

frame.time missing: 0
ip.src_host missing: 0
ip.dst_host missing: 0
arp.dst.proto_ipv4 missing: 0
arp.opcode missing: 0
arp.hw.size missing: 0
arp.src.proto_ipv4 missing: 0
icmp.checksum missing: 0
icmp.seq_le missing: 0
icmp.transmit_timestamp missing: 0
icmp.unused missing: 0
http.file_data missing: 0
http.content_length missing: 0
http.request.uri.query missing: 0
http.request.method missing: 0
http.referer missing: 0
http.request.full_uri missing: 0
http.request.version missing: 0
http.response missing: 0
http.tls_port missing: 0
tcp.ack missing: 0
tcp.ack_raw missing: 0
tcp.checksum missing: 0
tcp.connection.fin missing: 0
tcp.connection.rst missing: 0
tcp.connection.syn missing: 0
tcp.connection.synack missing: 0
tcp.dstport missing: 0
tcp.flags missing: 0
tcp.flags.ack missing: 0
tcp.len missing: 0
tcp.options missing: 0
tcp.payload missing: 0
tcp.seq missing: 0
tcp.srcport missing: 0
udp.port missing: 0
udp.stream missing: 0
udp.time_delta missing: 0
dns.qry.name missing: 0
dns

In [ ]:
# Attack_type 종류와 각 갯수 학인
counts = df['Attack_type'].value_counts()
print(counts)

Attack_type
Normal                   1615643
DDoS_UDP                  121568
DDoS_ICMP                 116436
SQL_injection              51203
Password                   50153
Vulnerability_scanner      50110
DDoS_TCP                   50062
DDoS_HTTP                  49911
Uploading                  37634
Backdoor                   24862
Port_Scanning              22564
XSS                        15915
Ransomware                 10925
MITM                        1214
Fingerprinting              1001
Name: count, dtype: int64


# **2. Feature engineering**

# **3. Building Model**

# **4. Data analysis**

# **github에 코드 저장**

https://github.com/yuchanleee/iot-ids-ml

제 깃허브 리포지토리 주소입니다.
일단 코랩에서 코드 자유롭게 추가하시고 잘 돌아가면 깃허브에 백업해놓으려고 합니다.  

제가 확인하고 올려도 상관없지만 혹시 수정하고 자기가 리포지토리에 직접 커밋하고 싶으시면 말해주세요 이메일로 초대 보내드릴게요